# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mhassantahir-afk/ML-Engineering-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one page (content_hash_id), for one client, on one specific day (report_date).

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# DESCRIBE just reads schema/metadata, doesn't scan the actual data
schema = con.execute(f"""
    DESCRIBE SELECT client_hash_id, content_hash_id, report_date FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").df()
print(schema)

       column_name column_type null   key default extra
0   client_hash_id     VARCHAR  YES  None    None  None
1  content_hash_id     VARCHAR  YES  None    None  None
2      report_date        DATE  YES  None    None  None


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features:
gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users,
ga4_engaged_sessions, ga4_total_engagement_sec, scroll_events

### Context:
report_date, content_hash_id, client_hash_id

Availability flags (used to filter, not as features): client_has_gsc, client_has_ga4,
gsc_data_available, ga4_data_available

### Label:
declining_flag: TRUE if a page's gsc_impressions dropped 20% or more from the first half of
the month to the second half.

first_half = sum of gsc_impressions from day 1 to day 15 of the month
second_half = sum of gsc_impressions from day 16 to end of month
pct_change = (second_half - first_half) / first_half * 100
declining_flag = TRUE if pct_change <= -20, else FALSE

Computed only on rows where gsc_data_available IS TRUE, and only for pages with nonzero
first_half impressions (a zero-impression starting point makes percent change undefined).

### Excluded:
gsc_sum_position (redundant with gsc_avg_position, which already captures the same signal.)

sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid,
sessions_ai (excluded because Lane 2 focuses on overall page decline, not traffic-source
attribution.)

ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other (excluded
because AI-referral data is extremely sparse (only 30,177 rows have any AI sessions, out of
78.8M total rows), too thin to be a reliable feature at this stage.)

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

result = con.execute(f"""
    WITH daily AS (
        SELECT
            content_hash_id,
            client_hash_id,
            report_date,
            gsc_impressions,
            CASE WHEN report_date < DATE '2026-03-16' THEN 'first_half' ELSE 'second_half' END AS period
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
    ),
    agg AS (
        SELECT
            content_hash_id,
            client_hash_id,
            period,
            SUM(gsc_impressions) AS total_impressions
        FROM daily
        GROUP BY content_hash_id, client_hash_id, period
    ),
    pivoted AS (
        SELECT
            content_hash_id,
            client_hash_id,
            MAX(CASE WHEN period = 'first_half' THEN total_impressions ELSE 0 END) AS first_half,
            MAX(CASE WHEN period = 'second_half' THEN total_impressions ELSE 0 END) AS second_half
        FROM agg
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT *,
        (second_half - first_half) * 1.0 / NULLIF(first_half, 0) * 100 AS pct_change,
        CASE WHEN (second_half - first_half) * 1.0 / NULLIF(first_half, 0) * 100 <= -20 THEN TRUE ELSE FALSE END AS declining_flag
    FROM pivoted
    WHERE first_half > 0
""").df()

print(result.head())
print(f"\nTotal pages: {len(result)}")
print(f"Declining pages: {result['declining_flag'].sum()} ({result['declining_flag'].mean()*100:.1f}%)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id           client_hash_id  first_half  second_half  \
0  content_b7e512995f79d5a6  client_73cda7b4e4f265ea       429.0        711.0   
1  content_a7da352b73b02668  client_73cda7b4e4f265ea      2440.0       2504.0   
2  content_1855a661b4d36130  client_73cda7b4e4f265ea       240.0        189.0   
3  content_d720dde3701523c0  client_73cda7b4e4f265ea        48.0         84.0   
4  content_91ffe8aef1f8c426  client_73cda7b4e4f265ea       129.0         83.0   

   pct_change  declining_flag  
0   65.734266           False  
1    2.622951           False  
2  -21.250000            True  
3   75.000000           False  
4  -35.658915            True  

Total pages: 151981
Declining pages: 50000 (32.9%)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1 Grain:**  Zero Rows Returned, which signals that One Page = One Row per Day per Client and that there are no duplicate rows.

**Query 2 Row count + date span:** shows the number of pages avaliable within a time frame of a month. This slice contains 9,841,378 rows, spanning exactly
2026-03-01 to 2026-03-31

**Query 3 Availability:** Shows the number of pages out of those pages within the time frame of a month that i usable. Here it shows Only 3,611,061 rows (36.7%) have gsc_data_available = TRUE, and
only 413,966 rows (4.2%) have ga4_data_available = TRUE. This confirms the unbalanced-panel
warning from the data documentation most rows in this month lack trustworthy search or
analytics data. Any feature or label built from GSC or GA4 columns must filter on
these flags first, or risk treating zero-filled placeholder data as real signal.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

grain_check = con.execute(f"""
    SELECT content_hash_id, client_hash_id, report_date, COUNT(*) AS row_count
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id, client_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Rows violating the grain: {len(grain_check)}")
print(grain_check)
print("If the above returns zero rows then it will be confirmed that there are no duplicate values in the dataset\n One Row = One Page per Client per Day\n\n")

count_and_span = con.execute(f"""
    SELECT COUNT(*) AS total_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print(count_and_span)


availability_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the grain: 0
Empty DataFrame
Columns: [content_hash_id, client_hash_id, report_date, row_count]
Index: []
If the above returns zero rows then it will be confirmed that there are no duplicate values in the dataset
 One Row = One Page per Client per Day


   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available_rows  ga4_available_rows
0     9841378           3611061.0            413966.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This Slice of Data cannot tell you about the clients without active tracking in the month of march 2026. only 36.7% of rows have gsc_data_available = TRUE and only 4.2% have
ga4_data_available = TRUE (Query 3), meaning any label or feature built here reflects a
specific, non-random subset of clients, not the full warehouse population.

This Slice has no visibility beyond the 31 days, whether the page was declining before, keeps declining or recovers immediately after.

This slice also cannot reliably distinguish a genuine sustained decline from short-term noise.
The declining_flag is based on a first-half vs. second-half split within a single 31-day month

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.